# Gemma 4 MT6985 Export (Google Colab)

This notebook bootstraps a Colab runtime to export `google/gemma-4-e2b-it` into an MT6985-targeted `.litertlm` bundle using the same helper scripts as this repo.

Before running:
- Use a High-RAM runtime.
- Add `HF_TOKEN` to Colab secrets or export it in the environment.
- Set `APP_REPO_URL` to the GitHub clone URL for the repo or branch that contains this notebook.
- If the notebook has not been merged to `main` yet, set `APP_REPO_REF` to the feature branch.

The notebook defaults to `CACHE_LENGTH=512` because that is the most realistic Colab-friendly starting point for the Gemma 4 export path.

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

APP_REPO_URL = os.environ.get("APP_REPO_URL", "https://github.com/dreef3/zvaka-app.git")
APP_REPO_REF = os.environ.get("APP_REPO_REF", "feat/litertlm-npu-setting")
LITERT_TORCH_REPO_URL = os.environ.get("LITERT_TORCH_REPO_URL", "https://github.com/google-ai-edge/litert-torch.git")
LITERT_TORCH_REF = os.environ.get("LITERT_TORCH_REF", "main")
MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-4-e2b-it")
MODEL_FAMILY = "gemma4"
CACHE_LENGTH = int(os.environ.get("CACHE_LENGTH", "512"))
PREFILL_LENGTHS = os.environ.get("PREFILL_LENGTHS", "128")
QUANTIZATION_RECIPE = os.environ.get("QUANTIZATION_RECIPE", "dynamic_wi4_afp32")

workspace = Path("/content/mt6985-colab")
app_root = workspace / "app"
src_root = workspace / "src"
litert_torch_root = src_root / "litert-torch"
output_root = workspace / "output" / f"gemma4-mt6985-cache{CACHE_LENGTH}"
log_path = workspace / "output" / f"gemma4-mt6985-cache{CACHE_LENGTH}.log"

def normalize_git_url(url: str) -> str:
    url = url.strip()
    if url.startswith("github.com/"):
        return f"https://{url}"
    if url.startswith("http://") or url.startswith("https://") or url.startswith("git@"):
        return url
    return url

def run(command: str) -> None:
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True)

if platform.machine() != "x86_64":
    raise RuntimeError(f"Expected x86_64 runtime, got {platform.machine()}")

APP_REPO_URL = normalize_git_url(APP_REPO_URL)

workspace.mkdir(parents=True, exist_ok=True)
print(f"Workspace: {workspace}")
print(f"Output directory: {output_root}")

In [ ]:
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN detected")
else:
    print("HF_TOKEN not set. Public access may still work, but authenticated access is recommended.")

In [ ]:
os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:{os.environ['PATH']}"
run("apt-get update -y")
run("apt-get install -y git")
run("curl -LsSf https://astral.sh/uv/install.sh | sh")

if app_root.exists():
    run(f"rm -rf {app_root}")
if litert_torch_root.exists():
    run(f"rm -rf {litert_torch_root}")

src_root.mkdir(parents=True, exist_ok=True)
run(f"git clone --depth 1 --branch {APP_REPO_REF} {APP_REPO_URL} {app_root}")
run(f"git clone --depth 1 --branch {LITERT_TORCH_REF} {LITERT_TORCH_REPO_URL} {litert_torch_root}")

In [ ]:
import shutil

inline_consts_path = litert_torch_root / "litert_torch" / "backend" / "inline_consts.py"
patched_inline_consts = app_root / "tools" / "colab" / "inline_consts.py"
if not patched_inline_consts.exists():
    raise RuntimeError(f"Missing patched inline_consts template: {patched_inline_consts}")
shutil.copyfile(patched_inline_consts, inline_consts_path)
print(f"Copied patched inline_consts.py to {inline_consts_path}")

In [ ]:
run(
    "bash -lc '"
    f"export PATH={Path.home() / '.local' / 'bin'}:$PATH && "
    f"cd {app_root} && "
    f"WORK_ROOT={app_root} SRC_ROOT={src_root} PYTHON_BIN=python3.12 ./tools/bootstrap_litert_export_env.sh"
    "'"
)

In [ ]:
output_root.parent.mkdir(parents=True, exist_ok=True)
run(
    "bash -lc '"
    f"export PATH={Path.home() / '.local' / 'bin'}:$PATH && "
    f"cd {app_root} && "
    f". .venv-litert/bin/activate && "
    f"PYTHONUNBUFFERED=1 python tools/build_mt6985_gemma3_litertlm.py --model {MODEL_ID} --model-family {MODEL_FAMILY} --cache-length {CACHE_LENGTH} --prefill-lengths {PREFILL_LENGTHS} --quantization-recipe {QUANTIZATION_RECIPE} --output-dir {output_root} --keep-intermediates 2>&1 | tee {log_path}"
    "'"
)

In [ ]:
run(f"find {output_root} -maxdepth 5 -type f | sort")
print(f"Log file: {log_path}")